# CubeBot RL — Phase 0: toolchain validation

Per `quadruped-rl-locomotion-plan.pdf`, Phase 0. Two milestones:

1. **Stock example runs untouched** — proves mujoco_playground + MJX + GPU work.
2. **Our URDF loads and a random policy twitches all 12 joints.**

Runtime → Change runtime type → **T4 GPU** before running.


## 0 · Install

In [ ]:
!pip install -q mujoco mujoco-mjx playground brax
!pip install -q mujoco_playground
import jax
print('jax', jax.__version__)
print('devices:', jax.devices())
assert jax.devices()[0].platform == 'gpu', 'No GPU — set Runtime type to T4 GPU'

## 1 · M1a — stock quadruped example, untouched

Train Go1 joystick for a deliberately short run. We are validating the
toolchain, not producing a policy — if this completes without error, MJX
training works on this machine.

In [ ]:
from mujoco_playground import registry

print([e for e in registry.ALL_ENVS if 'Go1' in e or 'Barkour' in e])

In [ ]:
from mujoco_playground import registry
from mujoco_playground.config import locomotion_params
from brax.training.agents.ppo import train as ppo
import functools, jax

ENV = 'Go1JoystickFlatTerrain'
env = registry.load(ENV)
cfg = locomotion_params.brax_ppo_config(ENV)

# cut it right down: toolchain check, not a real run
cfg.num_timesteps = 2_000_000
cfg.num_evals = 2

print(cfg)
train_fn = functools.partial(ppo.train, **dict(cfg))
make_inference, params, metrics = train_fn(environment=env,
                                           progress_fn=lambda s, m: print(s, m.get('eval/episode_reward')))
print('M1a PASS — stock MJX training completed')

## 2 · M1b — load the CubeBot model

Upload `assets/cubebot_12dof.xml` from the repo (generated by
`cubebot_rl/export_mjcf.py`, which reads the leg-study `params.yaml`, so it
carries the BOM-derived 0.743 kg mass and the real joint limits).

In [ ]:
from google.colab import files
up = files.upload()          # pick cubebot_12dof.xml
XML = list(up.keys())[0]
print('uploaded', XML)

In [ ]:
import mujoco
from mujoco import mjx
import jax, jax.numpy as jp

m = mujoco.MjModel.from_xml_path(XML)
print(f'nq={m.nq} nv={m.nv} nu={m.nu}  mass={m.body_mass.sum()*1000:.1f} g')
print('integrator:', mujoco.mjtIntegrator(m.opt.integrator).name, '(must be MJX-supported)')

mx = mjx.put_model(m)        # fails loudly if any feature is unsupported
print('MJX model built OK')

## 3 · M1c — random policy twitches all 12 joints, 4096 envs in parallel

In [ ]:
import numpy as np

N_ENV, N_STEP, DECIM = 4096, 300, 10
lo = jp.array(m.actuator_ctrlrange[:, 0]); hi = jp.array(m.actuator_ctrlrange[:, 1])

@jax.vmap
def init(rng):
    d = mjx.make_data(mx)
    return d.replace(qpos=jp.array(m.key_qpos[0]))

def ctrl_step(carry, _):
    d, rng, state = carry
    rng, k = jax.random.split(rng)
    tgt = jax.random.uniform(k, (m.nu,), minval=lo, maxval=hi) * 0.6
    state = 0.85 * state + 0.15 * tgt
    d = d.replace(ctrl=jp.clip(state, lo, hi))
    d = jax.lax.fori_loop(0, DECIM, lambda i, x: mjx.step(mx, x), d)
    return (d, rng, state), d.qpos

@jax.jit
@jax.vmap
def rollout(rng):
    d = mjx.make_data(mx).replace(qpos=jp.array(m.key_qpos[0]))
    (d, _, _), qs = jax.lax.scan(ctrl_step, (d, rng, jp.zeros(m.nu)), None, length=N_STEP)
    return qs

rngs = jax.random.split(jax.random.PRNGKey(0), N_ENV)
import time; t0 = time.time(); qs = rollout(rngs); qs.block_until_ready()
wall = time.time() - t0

steps = N_ENV * N_STEP * DECIM
print(f'{steps/wall/1e6:.1f} M physics steps/s  ({wall:.1f}s for {N_ENV} envs x {N_STEP} ctrl steps)')

jq = np.array(qs[:, :, 7:])           # drop the 7-dof free joint
sweep = np.degrees(jq.max(1) - jq.min(1)).mean(0)
names = [mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_JOINT, m.actuator_trnid[i,0]) for i in range(m.nu)]
for n, s in zip(names, sweep):
    print(f'  {n:12s} swept {s:6.1f} deg  {"yes" if s > 1 else "NO"}')
assert (sweep > 1).all(), 'some joints did not move'
print('\nM1 PASS — CubeBot runs under MJX and all 12 joints actuate.')

## Next

Phase 0 complete. **Phase 1 is the one that decides whether any of this
transfers**: bench-characterize one MG90S (step response, max speed under
load, deadband, stall) and fit a rate-limited position-tracking model with
30–80 ms latency. A policy trained on ideal position servos will not walk on
real MG90S hardware.